# Vector、Matrix 与运算

> 每个神经网络不过是 matrix 乘法加了一些额外的步骤。

**类型：** 动手实现
**语言：** Python, Julia
**前置要求：** Phase 1, Lesson 01（线性代数直觉）
**时间：** ~60 分钟

## 术语对照

- vector，向量
- matrix，矩阵
- transpose，转置
- determinant，行列式
- inverse，逆矩阵
- broadcasting，广播
- dense layer，全连接层
- activation，激活函数
- scalar，标量
- identity matrix，单位矩阵
## 关键术语

| 术语 | 人们怎么说 | 实际含义 |
|------|-----------|---------|
| Vector | "一个箭头" | 一个有序的数字列表。在 AI 中：高维空间中的一个点。 |
| Matrix | "一张数字表" | 一个线性 transformation。它把 vector 从一个空间映射到另一个空间。 |
| 矩阵乘法 | "把数字乘起来就是了" | 第一个 matrix 的每一行与第二个 matrix 的每一列之间的 dot product。顺序很重要。 |
| Transpose | "翻转它" | 交换行和列。把 m×n 的 matrix 变成 n×m。在 backpropagation 中至关重要。 |
| Determinant | "从 matrix 算出来的一个数" | 衡量 matrix 放大面积（2D）或体积（3D）的倍数。零意味着 transformation 压缩了一个维度。 |
| Inverse | "撤销这个 matrix" | 逆转该 transformation 的 matrix。仅当 determinant 不为零时才存在。 |
| Identity matrix | "那个无聊的 matrix" | 相当于数学中乘以 1 的 matrix。用于 residual connection（ResNet）。 |
| Broadcasting | "魔法般的 shape 修复" | 将较小的数组沿缺失维度重复以匹配较大的数组。 |
| 逐元素 | "常规乘法" | 将对应位置的数相乘。两个数组必须同 shape（或可 broadcast）。 |

## 扩展阅读

- [3Blue1Brown：线性代数的本质](https://www.3blue1brown.com/topics/linear-algebra) — 每个运算的直观视觉解释
- [NumPy broadcasting 文档](https://numpy.org/doc/stable/user/basics.broadcasting.html) — NumPy 遵循的精确规则
- [Stanford CS229 线性代数复习](http://cs229.stanford.edu/section/cs229-linalg.pdf) — ML 专用线性代数的简明参考

## 学习目标

- 构建一个 Matrix 类，支持逐元素运算、矩阵乘法、transpose、determinant 和 inverse
- 区分逐元素乘法和矩阵乘法，并解释各自的适用场景
- 仅使用从零实现的 Matrix 类实现一个单层 dense 神经网络（`relu(W @ x + b)`）
- 解释 broadcasting 规则以及 bias 加法在神经网络框架中的工作方式

## 问题

你想搭建一个神经网络。读代码时看到这行：


`output = activation(weights @ input + bias)`

那个 `@` 是矩阵乘法。`weights` 是一个 matrix。`input` 是一个 vector。如果你不知道这些运算在做什么，这行就是魔法。如果你知道，这就是一层 forward pass 的全部——三次运算。

你模型处理的每一张图片都是像素值的 matrix。每一个词 embedding 都是一个 vector。每一个神经网络的每一层都是一次 matrix transformation。你不可能在不精通 matrix 运算的情况下构建 AI 系统，就像你不可能不理解变量就写代码一样。

这节课从零开始建立这种熟练度。

## 概念

### Vector：有序的数字列表

一个 vector 是一串有方向和 magnitude 的数字。在 AI 中，vector 代表数据点、特征或参数。


`v = [3, 4]` — 一个 2D vector  
`w = [1, 0, -2]` — 一个 3D vector

2D vector `[3, 4]` 指向平面上的坐标 (3, 4)。它的长度（magnitude）是 5（3-4-5 三角形）。

### Matrix：数字网格

一个 matrix 是 2D 的网格。有行和列。一个 m×n 的 matrix 有 m 行 n 列。


```
A = | 1  2  3 |     -- 2×3 matrix（2 行，3 列）
    | 4  5  6 |
```

在神经网络中，weight matrix 把输入 vector 变换为输出 vector。一个有 784 个输入和 128 个输出的层使用 128×784 的 weight matrix。

### 为什么 shape 重要

矩阵乘法有严格规则：`(m × n) @ (n × p) = (m × p)`。内层维度必须匹配。


```
(128 × 784) @ (784 × 1) = (128 × 1)
   weights       input        output

内层维度：784 = 784  ——合法
```

如果你在 PyTorch 中遇到 shape mismatch 错误，就是这个原因。

### 运算速查表

| 运算 | 做什么 | 神经网络中的使用 |
|------|--------|----------------|
| 加法 | 逐元素相加 | 给输出加上 bias |
| Scalar 乘法 | 缩放每个元素 | learning rate × gradient |
| 矩阵乘法 | 变换 vector | 层的 forward pass |
| Transpose | 翻转行和列 | Backpropagation |
| Determinant | 单个数值概括 | 检查可逆性 |
| Inverse | 撤销一次变换 | 求解线性系统 |
| Identity | 不做任何事的 matrix | 初始化、residual connection |

### 逐元素乘法 vs 矩阵乘法

这个区别经常绊倒初学者。

逐元素：将对应位置的数相乘。两个 matrix 必须同 shape。


```
| 1  2 |   | 5  6 |   | 5  12 |
| 3  4 | * | 7  8 | = | 21 32 |
```

矩阵乘法：行的 dot product 与列。内层维度必须匹配。


```
| 1  2 |   | 5  6 |   | 1*5+2*7  1*6+2*8 |   | 19  22 |
| 3  4 | @ | 7  8 | = | 3*5+4*7  3*6+4*8 | = | 43  50 |
```

不同的运算，不同的结果，不同的规则。

### Broadcasting

当你把一个 bias vector 加到一个输出 matrix 上时，shape 不匹配。Broadcasting 会拉伸较小的数组以适配较大的。


```
| 1  2  3 |   +   [10, 20, 30]
| 4  5  6 |
```

Broadcasting 将 vector 按行拉伸：

```
| 1  2  3 |   | 10  20  30 |   | 11  22  33 |
| 4  5  6 | + | 10  20  30 | = | 14  25  36 |
```

每个现代框架都自动做这件事。理解它能避免你以为 shape 不对、但代码却正常运行时的困惑。

## 动手实现

### 第 1 步：Vector 类


In [44]:
class Vector:
    def __init__(self, data):
        self.data = list(data)
        self.size = len(self.data)

    def __repr__(self):
        return f"Vector({self.data})"

    def __add__(self, other):
        return Vector([a + b for a, b in zip(self.data, other.data)])

    def __sub__(self, other):
        return Vector([a - b for a, b in zip(self.data, other.data)])

    def __mul__(self, scalar):
        return Vector([x * scalar for x in self.data])

    def dot(self, other):
        return sum(a * b for a, b in zip(self.data, other.data))

    def magnitude(self):
        return sum(x ** 2 for x in self.data) ** 0.5


### dot 是怎么做到一一对应的？

**数学形式：**

两个向量靠**索引**来配对：

$$\mathbf{v} = [v_1, v_2, v_3], \quad \mathbf{w} = [w_1, w_2, w_3]$$

| 索引 $i$ | $\mathbf{v}$ | $\mathbf{w}$ | 配对 | 相乘 |
|----------|-------------|-------------|------|------|
| 0 | $v_1$ | $w_1$ | $(v_1, w_1)$ | $v_1 w_1$ |
| 1 | $v_2$ | $w_2$ | $(v_2, w_2)$ | $v_2 w_2$ |
| 2 | $v_3$ | $w_3$ | $(v_3, w_3)$ | $v_3 w_3$ |

最后求和：

$$\mathbf{v} \cdot \mathbf{w} = v_1 w_1 + v_2 w_2 + v_3 w_3$$

**代码写成显式循环就是：**

```python
def dot(self, other):
    result = 0
    for i in range(self.size):      # i = 0, 1, 2, ...
        a = self.data[i]             # v 的第 i 个元素
        b = other.data[i]            # w 的第 i 个元素
        result += a * b              # vi × wi，累加
    return result
```

`zip` 那行 `sum(a * b for a, b in zip(...))` 和上面完全等价，只是把索引 `i` 藏起来了而已。

具体例子：

```
v = [3, 4], w = [1, 2]

i=0: a=3, b=1 → 3×1 = 3
i=1: a=4, b=2 → 4×2 = 8
                   result = 3 + 8 = 11

v · w = 11
```



### 第 2 步：带核心运算的 Matrix 类


In [45]:
class Matrix:
    def __init__(self, data):
        self.data = [list(row) for row in data]
        self.rows = len(self.data)
        self.cols = len(self.data[0])
        self.shape = (self.rows, self.cols)

    def __repr__(self):
        rows_str = "\n  ".join(str(row) for row in self.data)
        return f"Matrix({self.shape}):\n  {rows_str}"

    def __add__(self, other):
        return Matrix([
            [self.data[i][j] + other.data[i][j] for j in range(self.cols)]
            for i in range(self.rows)
        ])

    def __sub__(self, other):
        return Matrix([
            [self.data[i][j] - other.data[i][j] for j in range(self.cols)]
            for i in range(self.rows)
        ])

    def scalar_multiply(self, scalar):
        return Matrix([
            [self.data[i][j] * scalar for j in range(self.cols)]
            for i in range(self.rows)
        ])

    def element_wise_multiply(self, other): # 逐元素乘法
        return Matrix([
            [self.data[i][j] * other.data[i][j] for j in range(self.cols)]
            for i in range(self.rows)
        ])

    def matmul(self, other):
        if self.cols != other.rows:
            raise ValueError(
                f"维度不匹配：（{self.rows}x{self.cols}） @ "
                f"{other.rows}x{self.cols}"
                f"内层维度{self.cols} ≠ {other.rows}"
            )
        return Matrix([
            [
                sum(self.data[i][k] * other.data[k][j] for k in range(self.cols))
                for j in range(other.cols)
            ]
            for i in range(self.rows)
        ])

    def transpose(self):
        return Matrix([
            [self.data[j][i] for j in range(self.rows)]
            for i in range(self.cols)
        ])

    def determinant(self):
        if self.shape == (1, 1):
            return self.data[0][0]
        if self.shape == (2, 2):
            return self.data[0][0] * self.data[1][1] - self.data[0][1] * self.data[1][0]
        det = 0
        for j in range(self.cols):
            # 构造minor：删除第0行 + 第j行
            minor = Matrix([
                [self.data[i][k] for k in range(self.cols) if k != j]
                for i in range(1, self.rows)
            ])
            det += ((-1) ** j) * self.data[0][j] * minor.determinant()
        return det

    def inverse_2x2(self):
        det = self.determinant()
        if det == 0:
            raise ValueError("Matrix 是奇异矩阵，逆不存在")
        return Matrix([
            [self.data[1][1] / det, -self.data[0][1] / det],
            [-self.data[1][0] / det, self.data[0][0] / det]
        ])

    @staticmethod
    def identity(n):
        return Matrix([
            [1 if i == j else 0 for j in range(n)]
            for i in range(n)
        ])
    
    def inverse_3x3(self):
        det = self.determinant()
        if det == 0:
            raise ValueError("Matrix 是奇异矩阵，逆不存在")
        
        # 计算9个cofactor
        cofactors = []
        for i in range(3):
            cofactor_row = []
            for j in range(3):
                # 构造去掉第i行、第j列的2x2 minor
                minor_data = []
                for r in range(3):
                    if r == i:
                        continue
                    row_vals = []
                    for c in range(3):
                        if c == j:
                            continue
                        row_vals.append(self.data[r][c])
                    minor_data.append(row_vals)
                minor = Matrix(minor_data)
                cofactor = ((-1) ** (i + j)) * minor.determinant()
                cofactor_row.append(cofactor)
            cofactors.append(cofactor_row)

        result_data = []
        for i in range(3):
            row = []
            for j in range(3):
                row.append(cofactors[j][i] / det)
            result_data.append(row)
        
        return Matrix(result_data)


> 🔧 `inverse_3x3` 定义如上 ↑。测试代码见下方 [练习 2](#)。



### 关键区分：数学表示 vs 代码存储

**铁律：`self.data[i]` 永远是第 $i$ 行。Python 的列表套列表没有"列"的概念。**

| 矩阵 | 数学写法 | `self.data` 存储 | `for row in m.data` 遍历出 |
|------|---------|-----------------|--------------------------|
| $2 \times 3$ | $\begin{bmatrix} 1 & 2 & 3 \\ 4 & 5 & 6 \end{bmatrix}$ | `[[1,2,3], [4,5,6]]` | `[1,2,3]` 是第 0 行 / `[4,5,6]` 是第 1 行 |
| $3 \times 1$ 列向量 | $\begin{bmatrix} 0.5 \\ 0.8 \\ 0.2 \end{bmatrix}$ | `[[0.5], [0.8], [0.2]]` | `[0.5]` 是第 0 行 / `[0.8]` 是第 1 行 / `[0.2]` 是第 2 行 |
| $1 \times 3$ 行向量 | $\begin{bmatrix} 1 & 2 & 3 \end{bmatrix}$ | `[[1,2,3]]` | `[1,2,3]` 是唯一一行 |

即使数学上竖着写的列向量，代码里也是 3 个独立的一行（每行只有 1 个元素）。`m.data` 取出来的永远是**行优先**的二维列表。



### 第 3 步：看它跑起来


In [46]:
A = Matrix([[1, 2], [3, 4]])
B = Matrix([[5, 6], [7, 8]])

print("A + B =")
print(A + B)
print("A @ B =")
print(A.matmul(B))
print("A^T =")
print(A.transpose())
print("det(A) =", A.determinant())
print("A^-1 =")
print(A.inverse_2x2())

I = Matrix.identity(2)
print("A @ A^-1 =")
print(A.matmul(A.inverse_2x2()))

A + B =
Matrix((2, 2)):
  [6, 8]
  [10, 12]
A @ B =
Matrix((2, 2)):
  [19, 22]
  [43, 50]
A^T =
Matrix((2, 2)):
  [1, 3]
  [2, 4]
det(A) = -2
A^-1 =
Matrix((2, 2)):
  [-2.0, 1.0]
  [1.5, -0.5]
A @ A^-1 =
Matrix((2, 2)):
  [1.0, 0.0]
  [0.0, 1.0]


### 第 4 步：连接到神经网络


In [47]:
import random

inputs = Matrix([[0.5], [0.8], [0.2]])
weights = Matrix([
    [random.uniform(-1, 1) for _ in range(3)]
    for _ in range(2)
])
bias = Matrix([[0.1], [0.1]])

def relu_matrix(m):
    return Matrix([[max(0, val) for val in row] for row in m.data])

pre_activation = weights.matmul(inputs) + bias
output = relu_matrix(pre_activation)

print(f"Input shape: {inputs.shape}")
print(f"Weight shape: {weights.shape}")
print(f"Output shape: {output.shape}")
print("Output:")
print(output)

Input shape: (3, 1)
Weight shape: (2, 3)
Output shape: (2, 1)
Output:
Matrix((2, 1)):
  [0]
  [0.4602235741456423]


这就是一个 single dense layer：`output = relu(W @ x + b)`。每个神经网络的每个 dense 层做的就是这个。

## 实际使用

NumPy 用更少的代码和数量级更快的速度完成上述所有事情。


In [48]:
import numpy as np

A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])

print("A + B =\n", A + B)
print("A * B (逐元素) =\n", A * B)
print("A @ B (矩阵乘法) =\n", A @ B)
print("A^T =\n", A.T)
print("det(A) =", np.linalg.det(A))
print("A^-1 =\n", np.linalg.inv(A))
print("I =\n", np.eye(2))

inputs = np.random.randn(3, 1)
weights = np.random.randn(2, 3)
bias = np.array([[0.1], [0.1]])
output = np.maximum(0, weights @ inputs + bias)

print(f"\n神经网络层：{weights.shape} @ {inputs.shape} = {output.shape}")
print(f"Output:\n{output}")


A + B =
 [[ 6  8]
 [10 12]]
A * B (逐元素) =
 [[ 5 12]
 [21 32]]
A @ B (矩阵乘法) =
 [[19 22]
 [43 50]]
A^T =
 [[1 3]
 [2 4]]
det(A) = -2.0000000000000004
A^-1 =
 [[-2.   1. ]
 [ 1.5 -0.5]]
I =
 [[1. 0.]
 [0. 1.]]

神经网络层：(2, 3) @ (3, 1) = (2, 1)
Output:
[[2.47841062]
 [1.65076663]]


Python 中的 `@` 运算符调用 `__matmul__`。NumPy 用 C 和 Fortran 编写的优化 BLAS 例程来实现。同样的数学，快 100 倍。

NumPy 的 Broadcasting：


### 正态分布（`np.random.randn` 的原理）

**定义：** 标准正态分布 $\mathcal{N}(0, 1)$，均值为 $0$，标准差为 $1$。

**概率密度函数：**

$$f(x) = \frac{1}{\sqrt{2\pi}} \cdot e^{-\frac{x^2}{2}}$$

- $x=0$ 时 $f(0) \approx 0.399$ — 概率密度最大
- $x = \pm 1$ 时 $f(\pm 1) \approx 0.242$ — 衰减到约 60%
- $x = \pm 2$ 时 $f(\pm 2) \approx 0.054$ — 衰减到约 14%
- $x \to \pm\infty$ 时 $f(x) \to 0$ — 但永远不会到 0

**6 个关键特性：**

| 特性 | 解释 |
|------|------|
| 钟形曲线 | 中间高、两边低，关于 $x=0$ 对称 |
| 68-95-99.7 规则 | 68% 的值在 $[-1,1]$，95% 在 $[-2,2]$，99.7% 在 $[-3,3]$ |
| 均值 $\mu=0$ | 抽样的值以 0 为中心，正负各半 |
| 标准差 $\sigma=1$ | 衡量"散开"的程度。$\sigma$ 越大曲线越扁平 |
| 指数衰减 $e^{-x^2}$ | 远离 0 的值概率急剧下降 |
| 归一化 $\frac{1}{\sqrt{2\pi}}$ | 曲线下总面积 = 1 |

**和均匀分布的区别：**

| | 均匀分布 `uniform(-1,1)` | 正态分布 `randn` |
|---|---|---|
| 概率 | 每个值概率相等 | 中间概率大，两边概率小 |
| 范围 | 严格 $[-1, 1]$ | 无界，但 99.7% 在 $[-3, 3]$ |
| 初始化权重 | 太"均匀"，梯度不好 | 大部分权重小，少数较大 → 更适合训练 |



In [49]:
matrix = np.array([[1, 2, 3], [4, 5, 6]])
bias = np.array([10, 20, 30])
print(matrix + bias)


[[11 22 33]
 [14 25 36]]


NumPy 自动将 1D 的 bias 按行 broadcast 到所有行。每个神经网络框架都是这样处理 bias 加法的。

## 产出

本节课产出一个通过几何直觉教授 matrix 运算的 prompt。见 `outputs/prompt-matrix-operations.md`。

此处构建的 Matrix 类是我们在 Phase 3 Lesson 10 中构建 mini 神经网络框架的基础。

## 练习

1. **验证逆矩阵。** 计算 `A @ A.inverse_2x2()` 确认结果为单位 matrix。用三个不同的 2×2 matrix 试试。当 determinant 为零时会发生什么？

2. **实现 3×3 逆矩阵。** 用伴随矩阵法扩展 Matrix 类以计算 3×3 matrix 的逆。与 NumPy 的 `np.linalg.inv` 对比测试。

3. **构建一个两层网络。** 仅使用你的 Matrix 类（不用 NumPy），创建一个两层神经网络：输入 (3) → 隐藏层 (4) → 输出 (2)。初始化随机 weights，运行一次 forward pass，验证所有 shape 正确。


---

## 📝 练习题（来自 exs-xyp）


# 练习题

这里包含 3 道练习。请补全每个函数体，然后运行验证 cell。

## 练习 1：验证逆矩阵

用 3 个不同的 2×2 矩阵，验证 `A @ A.inverse_2x2()` 是否等于单位矩阵。测试一下 det ≈ 0 时会发生什么。

In [50]:
def verify_inverse(A):
    """
    验证 A @ A.inverse_2x2() 是否等于单位矩阵。

    返回 True/False，并打印中间结果。
    """
    # TODO: 自己实现
    I = Matrix.identity(2)
    reslut = A.matmul(A.inverse_2x2())

    for i in range(reslut.rows):
        for j in range(reslut.cols):
            diff = abs(reslut.data[i][j] - I.data[i][j])
            if diff > 1e-10:
                print(f"不匹配 @ ({i},{j}): {result.data[i][j]} vs {I.data[i][j]}")
                return False
    return True


print("=== 练习 1：verify_inverse ===")
print(verify_inverse(Matrix([[4, 7], [2, 6]])),   "← 应为 True")
print(verify_inverse(Matrix([[1, 2], [3, 4]])),   "← 应为 True")
print(verify_inverse(Matrix([[5, 3], [3, 2]])),   "← 应为 True")

# 测试奇异矩阵会发生什么
print("\n--- 奇异矩阵测试 ---")
try:
    S = Matrix([[1, 2], [2, 4]])  # det = 4-4=0
    print(verify_inverse(S))
except ValueError as e:
    print(f"捕获异常：{e}  ← 这是预期行为")

=== 练习 1：verify_inverse ===
True ← 应为 True
True ← 应为 True
True ← 应为 True

--- 奇异矩阵测试 ---
捕获异常：Matrix 是奇异矩阵，逆不存在  ← 这是预期行为


## 练习 2：实现 3×3 求逆

用伴随矩阵（adjugate）方法实现 3×3 求逆。提示：

1. 算 9 个 cofactor，每个是 2×2 子矩阵的行列式
2. 组成 cofactor 矩阵，转置得 adjugate
3. 除以 det

测试时和 `A @ A_inv` 对比。

> 🔧 `inverse_3x3` 已写入上方 [Matrix 类](#) cell，执行后即可调用。修改函数时请回到「第 2 步」cell。



In [ ]:
print("\n=== 练习 2：3×3 inverse ===")
A = Matrix([[1, 2, 3], [0, 1, 4], [5, 6, 0]])
print("A =")
print(A)
A_inv = A.inverse_3x3()
print("\nA^-1 =")
print(A_inv)
print("\nA @ A^-1 (应该是单位矩阵) =")
print(A.matmul(A_inv))


=== 练习 2：3×3 inverse ===
A =
Matrix((3, 3)):
  [1, 2, 3]
  [0, 1, 4]
  [5, 6, 0]

A^-1 =
Matrix((3, 3)):
  [-24.0, 18.0, 5.0]
  [20.0, -15.0, -4.0]
  [-5.0, 4.0, 1.0]

A @ A^-1 (应该是单位矩阵) =
Matrix((3, 3)):
  [1.0, 0.0, 0.0]
  [0.0, 1.0, 0.0]
  [0.0, 0.0, 1.0]


## 练习 3：搭建两层神经网络

只用你的 Matrix 类（不引入 NumPy），构造一个两层网络：

- 输入 3 → 隐藏层 4（ReLU）→ 输出 2
- 用随机权重和 0 偏置
- 跑一次 forward pass，打印每层的 shape 和输出

In [58]:
def two_layer_forward(x):
    """两层神经网络 forward pass"""
    random.seed(42)
    W1 = Matrix([[random.uniform(-1, 1) for _ in range(3)] for _ in range(4)])
    b1 = Matrix([[0] for _ in range(4)])
    W2 = Matrix([[random.uniform(-1, 1) for _ in range(4)] for _ in range(2)])
    b2 = Matrix([[0] for _ in range(2)])

    z1 = (W1.matmul(x)) + b1
    h1 = relu_matrix(z1)
    z2 = (W2.matmul(h1)) + b2
    return h1, z2


print("=== 练习 3：两层神经网络 ===")
random.seed(42)
x = Matrix([[0.5], [0.8], [0.2]])
h1, z2 = two_layer_forward(x)

print(f"输入: {x.shape}")
print(f"隐藏层输出 (ReLU): {h1.shape}")
print(h1)
print(f"\n输出: {z2.shape}")
print(z2)

print("\nLayer 1: (4×3) @ (3×1) + (4×1) → (4×1) → ReLU → (4×1)")
print("Layer 2: (2×4) @ (4×1) + (2×1) → (2×1)")

=== 练习 3：两层神经网络 ===
输入: (3, 1)
隐藏层输出 (ReLU): (4, 1)
Matrix((4, 1)):
  [0]
  [0.17224447578040714]
  [0]
  [0]

输出: (2, 1)
Matrix((2, 1)):
  [-0.10374710196454817]
  [0.03075104184877088]

Layer 1: (4×3) @ (3×1) + (4×1) → (4×1) → ReLU → (4×1)
Layer 2: (2×4) @ (4×1) + (2×1) → (2×1)


## 简单验证

In [55]:
# 跑几个基本检查，确保没有语法错误
v = Vector([1, 2])
print("Vector: v + v =", v + v)

A = Matrix([[1, 0], [0, 1]])
B = Matrix([[2, 0], [0, 2]])
print("Matrix: A + B =")
print(A + B)
print("Matrix: A @ B =")
print(A.matmul(B))
print("det(I) =", A.determinant())
print("I^-1 =")
print(A.inverse_2x2())

Vector: v + v = Vector([2, 4])
Matrix: A + B =
Matrix((2, 2)):
  [3, 0]
  [0, 3]
Matrix: A @ B =
Matrix((2, 2)):
  [2, 0]
  [0, 2]
det(I) = 1
I^-1 =
Matrix((2, 2)):
  [1.0, 0.0]
  [0.0, 1.0]


In [ ]:
# ========================================
# 广播演示（不涉及任何 TODO，纯观察）
# ========================================

# 情况 A：形状相同 — 不需要广播
import numpy as np

A = Matrix([[1, 2, 3],
            [4, 5, 6]])   # 2行 3列

B = Matrix([[10, 20, 30],
            [40, 50, 60]]) # 2行 3列

print("情况 A — 形状相同 (2,3) + (2,3)：")
print("A =")
print(A)
print("B =")
print(B)
print("A + B =")
print(A + B)
print()

# 情况 B：偏置只有 1 行 — 需要广播
bias = Matrix([[100, 200, 300]])   # 1行 3列

print("情况 B — 广播 (2,3) + (1,3)：")
print("A (2,3) =")
print(A)
print("bias (1,3) =")
print(bias)
print("你想做的事：把 bias 这 1 行复制成 2 行，然后加")
print("广播自动帮你做了这件事：")

# Matrix类中只支持一一对应的加法
A_up = np.array([[1,2,3],[4,5,6]])
bias_np = np.array([100,200,300])
print(A_up + bias_np)

情况 A — 形状相同 (2,3) + (2,3)：
A =
Matrix((2, 3)):
  [1, 2, 3]
  [4, 5, 6]
B =
Matrix((2, 3)):
  [10, 20, 30]
  [40, 50, 60]
A + B =
Matrix((2, 3)):
  [11, 22, 33]
  [44, 55, 66]

情况 B — 广播 (2,3) + (1,3)：
A (2,3) =
Matrix((2, 3)):
  [1, 2, 3]
  [4, 5, 6]
bias (1,3) =
Matrix((1, 3)):
  [100, 200, 300]
你想做的事：把 bias 这 1 行复制成 2 行，然后加
广播自动帮你做了这件事：
[[101 202 303]
 [104 205 306]]
